# Layered Image Vectorization with Saliency Refinement (Google Colab)

This notebook sets up and runs [`layered_vectorization_with_saliency_refinement`](https://github.com/EdoDiT/layered_vectorization_with_saliency_refinement) on Google Colab, including all fixes needed to build `diffvg` on modern Colab (CUDA 12 / Python 3.11+ / CMake) and download all required model checkpoints.

**Before starting:** Go to **Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4, L4, or A100)**.

## 1. Check GPU & Environment

In [ ]:
!nvidia-smi
!nvcc --version
!gcc --version | head -n 1

## 2. Clone the Repository

In [ ]:
%cd /content

REPO_URL = "https://github.com/EdoDiT/layered_vectorization_with_saliency_refinement.git"
BRANCH = "design_additions"  # Change to "main" if needed
REPO_DIR = "/content/layered_vectorization_with_saliency_refinement"

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 3. Install Python Dependencies

We keep Colab's preinstalled PyTorch (matching the Colab CUDA driver) and install the packages needed by `diffvg`, `LayeredVectorization`, and `TranSalNet`.
We also pin `cmake==3.26.4` because `BachiLi/diffvg` uses `FindCUDA`, which was removed in CMake 3.27+.

In [ ]:
!pip uninstall -y -q tensorflow jax jaxlib flax
!pip install -q "cmake==3.26.4"
!pip install -q \
    numpy==1.26.3 \
    opencv-python==4.10.0.84 \
    segment-anything==1.0 \
    diffusers==0.20.2 \
    huggingface-hub==0.25.0 \
    transformers==4.46.0 \
    accelerate==1.0.1 \
    svgwrite svgpathtools cssutils numba torch-tools visdom \
    scikit-image scipy pyyaml tqdm pillow gdown

!cmake --version

## 4. Clone, Patch, and Build `diffvg` (CUDA)

Building `BachiLi/diffvg` on modern Colab requires three fixes:
1. **Update `pybind11` submodule** to `v2.13.6` (the bundled 2020 `pybind11` fails to compile on Python 3.11/3.12).
2. **Remove the bundled `thrust` include** from `CMakeLists.txt` so `nvcc` uses CUDA 12's built-in Thrust instead of the incompatible 2020 submodule.
3. **Upgrade C++ standard from `c++11`/`c++14` to `c++17`** (required by CUDA 12 Thrust and modern GCC).

In [ ]:
%cd /content
!rm -rf /content/diffvg
!git clone https://github.com/BachiLi/diffvg.git /content/diffvg
%cd /content/diffvg
!git submodule update --init --recursive

# 1. Update pybind11 for Python 3.11/3.12 compatibility
!git -C /content/diffvg/pybind11 fetch --tags
!git -C /content/diffvg/pybind11 checkout v2.13.6

# 2 & 3. Patch CMakeLists.txt for CUDA 12 + C++17
from pathlib import Path
import re

cmake_path = Path("/content/diffvg/CMakeLists.txt")
txt = cmake_path.read_text()

# Use CUDA toolkit's native Thrust instead of the outdated submodule
txt = re.sub(r'^.*include_directories\(.*thrust.*\).*$', '# (patched) use system CUDA thrust', txt, flags=re.MULTILINE)
# Upgrade C++ standard to C++17 for CUDA 12 + modern GCC
txt = txt.replace("-std=c++11", "-std=c++17").replace("-std=c++14", "-std=c++17")
txt = re.sub(r'set\(CMAKE_CXX_STANDARD\s+1[14]\)', 'set(CMAKE_CXX_STANDARD 17)', txt)
if "CMAKE_CXX_STANDARD 17" not in txt:
    txt = "set(CMAKE_CXX_STANDARD 17)\nset(CMAKE_CUDA_STANDARD 17)\n" + txt

cmake_path.write_text(txt)
print("Patched /content/diffvg/CMakeLists.txt successfully.")

In [ ]:
%cd /content/diffvg
# Remove poetry pyproject.toml so pip uses setup.py (which builds the CMake/CUDA extension)
!rm -f /content/diffvg/pyproject.toml
!DIFFVG_CUDA=1 pip install --no-build-isolation --no-deps -v .

# cd out of /content/diffvg and clear any cached failed imports before verifying
%cd /content
import sys, torch
for m in [k for k in sys.modules if k == "diffvg" or k.startswith("pydiffvg")]:
    del sys.modules[m]
import pydiffvg
pydiffvg.set_use_gpu(torch.cuda.is_available())
print("pydiffvg installed successfully! GPU enabled:", pydiffvg.get_use_gpu())

## 5. Download Required Model Checkpoints

The pipeline requires 3 local weight files (plus Stable Diffusion v1.5, which is auto-downloaded from Hugging Face on first run):
1. `sam_vit_h_4b8939.pth` — Segment Anything ViT-H
2. `TranSalNet/pretrained_models/TranSalNet_Res.pth` — TranSalNet saliency model
3. `TranSalNet/pretrained_models/resnet50-0676ba61.pth` — ResNet-50 ImageNet backbone used by TranSalNet

In [ ]:
%cd /content/layered_vectorization_with_saliency_refinement/LayeredVectorization
!mkdir -p TranSalNet/pretrained_models

# 1. SAM ViT-H checkpoint (~2.4 GB)
!wget -c -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O sam_vit_h_4b8939.pth

# 2. TranSalNet_Res checkpoint (from TranSalNet README Google Drive link)
!gdown 14czAAQQcRLGeiddPOM6AaTJTieu6QiHy -O TranSalNet/pretrained_models/TranSalNet_Res.pth

# 3. ResNet-50 ImageNet backbone (from PyTorch official mirror)
!wget -c -q --show-progress https://download.pytorch.org/models/resnet50-0676ba61.pth -O TranSalNet/pretrained_models/resnet50-0676ba61.pth

!ls -lh sam_vit_h_4b8939.pth TranSalNet/pretrained_models/

## 6. Patch Repo for Colab

Two quick fixes are needed in the repo files before running:
1. **`sds_image_simplicity.py`**: Replace the deprecated `runwayml/stable-diffusion-v1-5` Hugging Face ID (which now returns 401/404) with the official mirror `stable-diffusion-v1-5/stable-diffusion-v1-5`.
2. **`main.py`**: Restore CLI argument parsing in `if __name__ == "__main__":` (currently commented out in favor of a hardcoded `/home/sagemaker-user/...` loop).

In [ ]:
from pathlib import Path

lv_dir = Path("/content/layered_vectorization_with_saliency_refinement/LayeredVectorization")

# 1. Fix Stable Diffusion v1.5 Hugging Face repo ID
sds_path = lv_dir / "sds_image_simplicity.py"
sds_txt = sds_path.read_text()
sds_txt = sds_txt.replace(
    'model_id = "runwayml/stable-diffusion-v1-5"',
    'model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"',
)
sds_path.write_text(sds_txt)
print("Patched sds_image_simplicity.py")

# 2. Restore CLI argument parsing in main.py
main_path = lv_dir / "main.py"
main_txt = main_path.read_text()
marker = 'if __name__ == "__main__":'
assert marker in main_txt, "Could not find __main__ block in main.py"
main_txt = main_txt.split(marker)[0] + '''if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="layered_image_vectorization")
    parser.add_argument("-c", "--config", type=str, default="./config/base_config.yaml", help="YAML/YML file for configuration.")
    parser.add_argument("-timg", "--target_image", default="./target_imgs/robot.png", type=str)
    parser.add_argument("-fsn", "--file_save_name", type=str, default="robot", help="Files save name.")

    args = parser.parse_args()
    args = load_config(args.config, args)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    init_diffvg(device=device)
    layered_vectorization(args, device)
'''
main_path.write_text(main_txt)
print("Patched main.py")

## 7. Run Layered Vectorization

Pick any image from `target_imgs/` (or upload your own to `LayeredVectorization/target_imgs/`) and run!

In [ ]:
%cd /content/layered_vectorization_with_saliency_refinement/LayeredVectorization
!ls target_imgs/

In [ ]:
%cd /content/layered_vectorization_with_saliency_refinement/LayeredVectorization

import os
import subprocess
from pathlib import Path

# Selected ~half of target_imgs (comment/uncomment to customize)
IMAGES = [
    ("target_imgs/resized_beach_vendor.jpg", "beach_vendor_colab"),
    ("target_imgs/resized_nizar.jpg", "nizar_colab"),
    ("target_imgs/resized_berengo_coffee.jpg", "berengo_coffee_colab"),
    ("target_imgs/resized_berengo_couple.jpg", "berengo_couple_colab"),
    ("target_imgs/resized_ghirri_slide.jpg", "ghirri_slide_colab"),
    ("target_imgs/resized_ghirri_lake.jpg", "ghirri_lake_colab"),
    ("target_imgs/resized_hiker.jpg", "hiker_colab"),
    ("target_imgs/resized_lake_house.jpg", "lake_house_colab"),
    ("target_imgs/resized_still_life.jpg", "still_life_colab"),
    ("target_imgs/resized_sunbathing.jpg", "sunbathing_colab"),
    ("target_imgs/resized_vietnam.jpg", "vietnam_colab"),
]

env = os.environ.copy()
env.update({
    "USE_TORCH": "1",
    "USE_TF": "0",
    "USE_FLAX": "0",
    "TRANSFORMERS_NO_TF": "1",
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
})

for idx, (target_img, save_name) in enumerate(IMAGES, 1):
    final_svg = Path("workdir") / save_name / "final.svg"
    if final_svg.exists():
        print(f"[{idx}/{len(IMAGES)}] Skipping {save_name} (already done: {final_svg})")
        continue
    print(f"\n=== [{idx}/{len(IMAGES)}] Running {target_img} -> workdir/{save_name} ===")
    subprocess.run(
        [
            "python", "main.py",
            "--config", "config/base_config.yaml",
            "--target_image", target_img,
            "--file_save_name", save_name,
        ],
        env=env,
        check=True,
    )

## 8. Visualize & Save Results

In [ ]:
import re
from IPython.display import display, HTML
from pathlib import Path

workdir = Path("/content/layered_vectorization_with_saliency_refinement/LayeredVectorization/workdir")

def _svg_card(title: str, svg_path: Path) -> str:
    if not svg_path or not svg_path.exists():
        return ""
    svg_txt = svg_path.read_text()
    n_paths = svg_txt.count("<path")
    if "viewBox" not in svg_txt:
        svg_txt = re.sub(r'width="(\d+)"\s+height="(\d+)"', r'viewBox="0 0 \1 \2" width="256" height="256"', svg_txt, count=1)
    return (
        "<div style='text-align:center; width:260px;'>"
        f"<div style='font-weight:bold; margin-bottom:4px;'>{title} ({n_paths} paths)</div>"
        f"{svg_txt}"
        "</div>"
    )

for _, save_name in IMAGES:
    out_dir = workdir / save_name
    if not out_dir.exists():
        continue
    struct_svgs = sorted((out_dir / "struct_svgs").glob("*.svg"), key=lambda p: int(p.stem))
    before_merge = struct_svgs[-1] if struct_svgs else None

    cards = [
        _svg_card("Before struct merge (49.svg)", before_merge),
        _svg_card("After struct merge", out_dir / "struct_merged.svg"),
        _svg_card("Concave simplified", out_dir / "concave_simplified.svg"),
        _svg_card("Color adjusted", out_dir / "color-adjusted.svg"),
        _svg_card("Final", out_dir / "final.svg"),
    ]
    cards_html = "".join(c for c in cards if c)
    if cards_html:
        display(HTML(
            f"<h3>{save_name}</h3>"
            f"<div style='display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start;'>{cards_html}</div>"
        ))

In [ ]:
# Zip ONLY the 5 key SVGs per image: 49.svg (before merge), struct_merged.svg,
# concave_simplified.svg, color-adjusted.svg, and final.svg
import zipfile
from pathlib import Path

workdir = Path("/content/layered_vectorization_with_saliency_refinement/LayeredVectorization/workdir")
zip_path = Path("/content/vectorization_results.zip")

KEY_SVGS = [
    "struct_merged.svg",
    "concave_simplified.svg",
    "color-adjusted.svg",
    "final.svg",
]

file_count = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for _, save_name in IMAGES:
        out_dir = workdir / save_name
        if not out_dir.exists():
            continue
        # 1. Last struct SVG before merging (49.svg by default)
        struct_svgs = sorted((out_dir / "struct_svgs").glob("*.svg"), key=lambda p: int(p.stem))
        if struct_svgs:
            zf.write(struct_svgs[-1], arcname=f"{save_name}/{struct_svgs[-1].name}")
            file_count += 1
        # 2-5. Merged, concave, color-adjusted, and final SVGs
        for svg_name in KEY_SVGS:
            svg_path = out_dir / svg_name
            if svg_path.exists():
                zf.write(svg_path, arcname=f"{save_name}/{svg_name}")
                file_count += 1

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Saved {file_count} SVGs to {zip_path} ({size_mb:.2f} MB)")

# Option A (browser Colab): download directly
# from google.colab import files
# files.download(str(zip_path))

# Option B (VS Code Colab extension): copy to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !cp {zip_path} /content/drive/MyDrive/